# 03 — Ablation Exp 1–6 (proposal 3.3.3) → RQ1

In [1]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import numpy as np
import pandas as pd

from src import config
from src.utils import set_seed, save_fig
set_seed()  # fix all RNGs -- reproducibility

Six conditions, identical classifier + protocol. TF-IDF sparse + dense → `sparse.hstack` in `features.assemble`. Add LinearSVM (all), MultinomialNB (TF-IDF conditions only — non-negative).

In [2]:
from src import data, features, modeling
clean = data.load_or_build_clean(); splits = data.load_or_build_splits(clean)
y = clean[config.LABEL_COL].values

In [3]:
# Build every block ONCE, then assemble per experiment.
texts = clean['text']
X_tfidf, _ = features.build_tfidf(texts.iloc[splits['train']], texts)
sty = features.build_stylometric(texts)
sty_scaled, _ = features.scale_dense(sty.values, splits['train'])
bib = features.build_biber(texts)
bib_scaled, _ = features.scale_dense(bib.values, splits['train'])
emb = features.build_sbert(texts)   # cached

blocks = {
    'tfidf':       X_tfidf,
    'stylometric': sty_scaled,
    'biber':       bib_scaled,
    'sbert':       emb,
    'length':      clean[['log_token_count']].values,
}

In [5]:
# NB needs non-negative input everywhere -- only exp1 (pure TF-IDF + length,
# both non-negative) qualifies. Scaled dense blocks are mean-centered, so any
# experiment mixing them in would violate MultinomialNB's assumption even if
# it also contains tfidf (proposal 3.2: "TF-IDF-based conditions ... non-negative").
CLASSIFIERS_BY_EXP = {'exp1_tfidf': ['logreg', 'svm', 'nb']}
DEFAULT_CLASSIFIERS = ['logreg', 'svm']

ytr, yval = y[splits['train']], y[splits['val']]
results = {}
for exp_name, which in features.EXPERIMENTS.items():
    X = features.assemble(blocks, which)
    Xtr, Xval = X[splits['train']], X[splits['val']]
    for clf_name in CLASSIFIERS_BY_EXP.get(exp_name, DEFAULT_CLASSIFIERS):
        results[(exp_name, clf_name)] = modeling.train_and_evaluate(clf_name, Xtr, ytr, Xval, yval)

results_df = pd.DataFrame({
    (exp, clf): {'macro_f1': r.macro_f1, 'weighted_f1': r.weighted_f1, 'accuracy': r.accuracy}
    for (exp, clf), r in results.items()
}).T
results_df.index.names = ['experiment', 'classifier']
results_df

/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


: 

: 

Save the results table for the report.

In [ ]:
results_df.to_csv(config.ARTIFACTS / 'ablation_results.csv')

In [ ]:
from src import viz

pivot = results_df['macro_f1'].unstack('classifier').reindex(features.EXPERIMENTS.keys())

fig, ax = viz.new_fig(figsize=(9, 5))
n_clf = len(pivot.columns)
width = 0.8 / n_clf
x = np.arange(len(pivot))
for i, clf in enumerate(pivot.columns):
    vals = pivot[clf].values
    valid = ~np.isnan(vals)
    offset = (i - (n_clf - 1) / 2) * width
    bars = ax.bar(x[valid] + offset, vals[valid], width, label=clf,
                   color=viz.CATEGORICAL[i], zorder=3)
    ax.bar_label(bars, fmt='%.2f', fontsize=7, color=viz.INK_SECONDARY, padding=2)

ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=20, ha='right')
ax.set_ylabel('Macro-F1')
ax.set_ylim(0, 1)
ax.set_title('Ablation (Exp 1–6): Macro-F1 by feature condition and classifier')
ax.legend(frameon=False, fontsize=9, title=None)
fig.tight_layout()
save_fig(fig, 'ablation_macro_f1')